In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq
LN2 = np.log(2.0)

def lin(x):
    return np.exp2(x) - 1.0

files = sorted(glob.glob(f"{SLIM_DIR}/*_slim.parquet"))
parts = []
for f in files:
    roi = os.path.basename(f).replace("_slim.parquet", "")
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in GRMS if g in names]
    d = pd.read_parquet(f, columns=["class", "subclass"] + genes)
    d = d[[is_neuron_class(c) for c in d["class"]]].copy()
    for g in genes:
        d[g] = d[g].astype("float32")
    d.insert(0, "roi", roi)
    parts.append(d)
    print(roi, len(d), "neurons", flush=True)
df = pd.concat(parts, ignore_index=True)
for g in GRMS:
    if g not in df.columns:
        df[g] = np.float32(0)
df[GRMS] = df[GRMS].fillna(0)
for c in ("roi", "class", "subclass"):
    df[c] = df[c].astype("category")
thr = {g: float(np.median(df[g][df[g] > 0])) if (df[g] > 0).any() else np.inf for g in GRMS}
print("per-receptor positivity threshold (median log2 among detected neurons):", {g: round(v, 2) for g, v in thr.items()})

import itertools, re
MIN_GROUP = 100
MIN_VIOLIN = 30
ONLY_PAIRS = None  # e.g. [("Grm7", "Grm8")] to run just one

def family(sub, cls):
    s, c = str(sub), str(cls)
    if "GABA" not in c and "Glut" not in c and "Gaba" not in s and "Glut" not in s:
        return None
    if re.search(r"Pvalb", s, re.I): return "PV interneuron (GABA)"
    if re.search(r"Sst", s, re.I): return "SST interneuron (GABA)"
    if re.search(r"Vip", s, re.I): return "VIP interneuron (GABA)"
    if re.search(r"Lamp5", s, re.I): return "LAMP5 interneuron (GABA)"
    if re.search(r"Sncg", s, re.I): return "SNCG interneuron (GABA)"
    if re.search(r"D1", s) and re.search(r"STR", s): return "D1 MSN (GABA)"
    if re.search(r"D2", s) and re.search(r"STR", s): return "D2 MSN (GABA)"
    if "GABA" in c or "Gaba" in s: return "other GABAergic"
    if re.search(r"\bIT\b", s): return "IT Glut"
    if re.search(r"\bCT\b|\bNP\b|\bET\b|L6b", s): return "CT/NP/ET/L6b Glut"
    if re.search(r"\bDG\b|CA1|CA2|CA3|ProS|\bSUB\b", s): return "hippocampal Glut"
    return "other Glut"

smap = df[["subclass", "class"]].drop_duplicates()
smap["family"] = [family(a, b) for a, b in zip(smap["subclass"], smap["class"])]
smap.to_csv(f"{RES_DIR}/coexpression/type_map_subclass_to_family.csv", index=False)
df = df[df["roi"].isin(FOCUS_ROIS)].copy()
df["family"] = df["subclass"].astype(str).map(dict(zip(smap["subclass"].astype(str), smap["family"])))
df = df[df["family"].notna()]
FAMS = [f for f in ["PV interneuron (GABA)", "SST interneuron (GABA)", "VIP interneuron (GABA)", "LAMP5 interneuron (GABA)", "SNCG interneuron (GABA)",
                    "D1 MSN (GABA)", "D2 MSN (GABA)", "other GABAergic",
                    "IT Glut", "CT/NP/ET/L6b Glut", "hippocampal Glut", "other Glut"] if f in set(df["family"])]
regs = [r for r in FOCUS_ROIS if r in set(df["roi"])]
rcol = {r: plt.get_cmap("plasma")(0.08 + 0.8 * i / max(1, len(regs) - 1)) for i, r in enumerate(regs)}
fam_idx = df.groupby(["family", "roi"], observed=True).indices
rng = np.random.default_rng(0)
allrows = []
pairs = ONLY_PAIRS or list(itertools.combinations(GRMS, 2))
for A, B in pairs:
    both = (df[A].to_numpy() >= thr[A]) & (df[B].to_numpy() >= thr[B])
    gm = np.sqrt(lin(df[A].to_numpy().astype("float64")) * lin(df[B].to_numpy().astype("float64")))
    ygm = np.log2(gm + 1)
    fig, ax = plt.subplots(figsize=(22, 7))
    x, ticks, famticks = 0.0, [], []
    for fam in FAMS:
        x0 = x
        for r in regs:
            ix = fam_idx.get((fam, r))
            if ix is None or len(ix) < MIN_GROUP:
                x += 1
                continue
            m = both[ix]
            pct = 100 * m.mean()
            allrows.append((A, B, fam, r, len(ix), float(pct), int(m.sum()), float(ygm[ix][m].mean()) if m.any() else np.nan))
            if m.sum() >= MIN_VIOLIN:
                v = ygm[ix][m]
                vp = ax.violinplot([v], positions=[x], widths=0.85, showextrema=False)
                for b in vp["bodies"]:
                    b.set_facecolor(rcol[r]); b.set_edgecolor("k"); b.set_linewidth(0.3); b.set_alpha(0.85)
                s = rng.choice(v, min(120, len(v)), replace=False)
                ax.scatter(x + rng.uniform(-0.25, 0.25, len(s)), s, s=3, c="k", alpha=0.35, linewidths=0, zorder=3)
            ax.text(x, 1.01, f"{pct:.0f}%", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=7, rotation=90)
            ticks.append((x, r))
            x += 1
        famticks.append(((x0 + x - 1) / 2, fam))
        if fam != FAMS[-1]:
            ax.axvline(x + 0.0, color="0.8", lw=0.8)
        x += 1
    ax.set_xticks([t[0] for t in ticks])
    ax.set_xticklabels([t[1] for t in ticks], rotation=90, fontsize=8)
    for xc, fam in famticks:
        ax.text(xc, -0.17, fam.replace(" (", chr(10) + "("), transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=9)
    ax.set_ylabel(f"geometric mean of {A} and {B}\n(log2 of sqrt(linear {A} x linear {B}) + 1), co-expressing cells", fontsize=10)
    ax.set_title(f"{A} and {B} co-expression in single neurons, by cell type and brain region (Allen WMB, 10x)\nviolin = cells with both receptors above threshold (>= {MIN_VIOLIN} cells); dots = random cells; number on top = % of the group's cells co-expressing; color = region (x labels)", fontsize=10, pad=28)
    handles = [plt.Rectangle((0, 0), 1, 1, fc=rcol[r], ec="k", lw=0.3) for r in regs]
    ax.legend(handles, [f"{NAMES.get(r, r)} ({r})" for r in regs], frameon=False, fontsize=8, ncol=2, loc="upper right", bbox_to_anchor=(1.0, 1.0))
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    fig.savefig(f"{FIG_DIR}/ALL_pair_{A}_{B}_by_type.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(A, B, flush=True)
pd.DataFrame(allrows, columns=["gene_a", "gene_b", "family", "roi", "n_cells", "pct_coexpressing", "n_coexpressing", "mean_log2_geomean_coexpr"]).to_csv(f"{RES_DIR}/coexpression/pair_by_type_summary.csv", index=False)